##Installing Dependencies
RAG PIPELINE:
document loading
embedding models
vector database
llm model

In [ ]:
print("Installing")

!pip install -q \
    langchain \
    langchain-community \
    langchain-text-splitters \
    langchain-huggingface \
    langchain-groq \
    langchain-google-genai \
    langchain-openai \
    langchain-core \
    faiss-cpu \
    pypdf \
    sentence-transformers \
    transformers \
    torch \
    huggingface_hub \
    groq \
    langsmith \
    python-dotenv \
    tiktoken

print("Installation Completed ")


Installing
Installation Completed 



[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
LLM_Provider="google-genai"
LLM_Model="gemini-3.7-flash"
CORPUS_PATH="./zyro-dynamics-hr-corpus"

In [ ]:
import os
from dotenv import load_dotenv
load_dotenv(override=True)
if os.getenv("LLM_API_KEY"):
    print("GROQ API KEY is importend")
else:
    print("Not found")


GROQ API KEY is importend


In [ ]:
from langchain_community.document_loaders import PyPDFDirectoryLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
loader=PyPDFDirectoryLoader(CORPUS_PATH)
documents=loader.load()
print(f"Loaded {len(documents)} documents")

C:\Users\Bhagathkumar\AppData\Local\Temp\ipykernel_24044\2162728304.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFDirectoryLoader
c:\Users\Bhagathkumar\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loaded 39 documents


In [ ]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,
    chunk_overlap=100
)
chuncks=splitter.split_documents(documents)
print(f"Created {len(chuncks)}")

Created 107


#Embeddings Initalization
HuggingFaceEmbeddings

In [ ]:
from langchain_huggingface import HuggingFaceEmbeddings
embedding_model=HuggingFaceEmbeddings(
    model_name ="sentence-transformers/all-MiniLM-L6-v2"
)
print("Embeddings Done!")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2351.03it/s]


Embeddings Done!


## VectorDB+Retrival(building retrival pipeline)
FAISS Vector DB


In [ ]:
from langchain_community.vectorstores import FAISS
vectorstore=FAISS.from_documents(chuncks,embedding_model)#chunck of the data,model->Embedding models
#Retrival pipeline 
retriver = vectorstore.as_retriever(search_kwargs={"k":3}) 
print("Done") 

Done


## LLM Initalization 
LLM model - gemini

temperature(reason for getting differnt resposnse each time)

temperature=0.7
max_tokens=500

In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI
llm_model=ChatGoogleGenerativeAI(
    model="gemini-3.7-flash",
    google_api_key=os.getenv("LLM_API_KEY"),
    temperature=0.7,
    max_tokens=500
)
print(f"LLM Model {llm_model} initalized")

LLM Model metadata={'lc_versions': {'langchain-core': '1.6.3', 'langchain': '1.3.15', 'langchain-google-genai': '4.3.3'}} profile={} google_api_key=SecretStr('**********') model='gemini-3.7-flash' max_output_tokens=500 client=<google.genai.client.Client object at 0x00000168B340A7B0> default_metadata=() model_kwargs={} initalized


RAG CHAIN

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langsmith import traceable
RAG_PROMPT = ChatPromptTemplate.from_template(
    #define your prompt
    """You are an HR assistant. Answer the question using ONLY the context below .If answer isn't in the context,say you dont't have that information.
    context:{context},
    question:{question}"""
)
#all the info required for LLM Model to get the output
def foramt_docs(docs):
    return ".\n\n".join(d.page_content for d in docs)
#context :understanding about the question recieved from the user
@traceable(name="rag_chain")#using tracable to understand context ans question
def rag_chain(question:str):
    docs = retriver.invoke(question)
    context=foramt_docs(docs)
    chain=RAG_PROMPT | llm_model | StrOutputParser()
    answer=chain.invoke({"context":context,"question":question})
    return {"answer":answer}
print(RAG_PROMPT.input_variables)


['context', 'question']


##Guardials

In [ ]:
GUARDIAL_PROMPT=ChatPromptTemplate.from_template("""
You are a scope classifier for an HR assistant. Decide whether the question
below is something an HR assistant should answer (company leave policy,
reimbursement, code of conduct, or other internal HR topics) or something
out of scope (general knowledge, coding help, unrelated small talk, etc).
Respond with exactly one word: IN_SCOPE or OUT_OF_SCOPE.
question: {question}
""")
REFUSAL_MESSAGE = (
    "I'm an HR assistant and can only help with questions about company HR "
    "policies (leave, reimbursement, code of conduct, etc.). I don't have "
    "information to answer that question."
)
def ask_bot(question:str, *args, **kwargs):
    guardial_chain=GUARDIAL_PROMPT | llm_model | StrOutputParser()
    verdict=guardial_chain.invoke({"question":question}).strip().upper()
    if "OUT_OF_SCOPE" in verdict:
        return {'answer':REFUSAL_MESSAGE,"sources":[]}
    return rag_chain(question)
print("Guardial Initalized")

Guardial Initalized


In [ ]:
import pandas as pd

test_df = pd.read_csv("test.csv")

answers = []
for i, row in test_df.iterrows():
    q = row["question"]
    result = ask_bot(q)
    answers.append(result["answer"])
    print(f"[{i+1}/{len(test_df)}] {q}\n-> {result['answer']}\n{'-'*60}")

test_df["answer"] = answers

if "question_id" in test_df.columns:
    submission = test_df[["question_id", "answer"]]
else:
    test_df["question_id"] = [f"Q{i+1:02d}" for i in range(len(test_df))]
    submission = test_df[["question_id", "answer"]]

submission.to_csv("submission.csv", index=False)
submission.head()


In [ ]:
import os

print(os.listdir("/kaggle/input"))